# Hybrid Model Training and Evaluation

In [ ]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier


In [ ]:
df = pd.read_csv('../data/processed/clustered_career_stream_dataset.csv')


In [ ]:
target_col = 'recommended_stream'

X = df.select_dtypes(include=['int64','float64'])
y = df[target_col]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    print(name)
    print('Accuracy:', accuracy_score(y_test, preds))
    print(classification_report(y_test, preds))


In [ ]:
voting_model = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000)),
        ('nb', GaussianNB()),
        ('knn', KNeighborsClassifier(n_neighbors=5)),
        ('dt', DecisionTreeClassifier(random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
    ],
    voting='soft'
)

voting_model.fit(X_train_scaled, y_train)

hybrid_preds = voting_model.predict(X_test_scaled)

print('Hybrid Accuracy:', accuracy_score(y_test, hybrid_preds))
print(classification_report(y_test, hybrid_preds))

In [ ]:
joblib.dump(voting_model, '../models/hybrid_stream_recommendation_model.pkl')
joblib.dump(scaler, '../models/model_scaler.pkl')
joblib.dump(label_encoder, '../models/stream_label_encoder.pkl')

print('Saved hybrid model assets')